# 🔍 Analyse Exploratoire des Données Industrielles

## Dataset : AI4I 2020 Predictive Maintenance

Ce notebook analyse les patterns de pannes industrielles pour préparer notre système IoT de maintenance prédictive.

**Objectifs :**
1. 📊 Comprendre la distribution des variables capteurs
2. 🔥 Analyser les patterns de chaque type de panne
3. 📈 Identifier les corrélations critiques
4. 🎯 Extraire des seuils d'alerte réalistes
5. 🤖 Préparer les features pour le ML

In [ ]:
# Import des librairies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("📚 Librairies importées avec succès !")

In [ ]:
# Chargement des données
data_path = "../data/raw/ai4i2020_demo.csv"
scenarios_path = "../data/processed/failure_scenarios.csv"

# Dataset principal
df = pd.read_csv(data_path)
print(f"📊 Dataset chargé : {df.shape[0]} échantillons, {df.shape[1]} variables")

# Scénarios de pannes
scenarios = pd.read_csv(scenarios_path)
print(f"🔥 Scénarios de pannes : {len(scenarios)} cas extraits")

# Aperçu des données
print("\n🔍 Aperçu du dataset :")
display(df.head())

print("\n📈 Informations générales :")
display(df.info())

In [ ]:
# Analyse des pannes - Vue d'ensemble
failure_stats = {
    'Total échantillons': len(df),
    'Échantillons normaux': len(df[df['Machine failure'] == 0]),
    'Pannes totales': len(df[df['Machine failure'] == 1]),
    'Taux de panne global': f"{df['Machine failure'].mean():.3f} ({df['Machine failure'].mean()*100:.1f}%)"
}

print("🎯 STATISTIQUES GÉNÉRALES")
print("=" * 50)
for key, value in failure_stats.items():
    print(f"{key:<25}: {value}")

# Détail par type de panne
failure_types = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
failure_names = {
    'TWF': 'Tool Wear Failure',
    'HDF': 'Heat Dissipation Failure', 
    'PWF': 'Power Failure',
    'OSF': 'Overstrain Failure',
    'RNF': 'Random Failure'
}

print("\n🔥 RÉPARTITION PAR TYPE DE PANNE")
print("=" * 50)
for failure_type in failure_types:
    count = df[failure_type].sum()
    rate = count / len(df) * 100
    print(f"{failure_names[failure_type]:<25}: {count:3d} cas ({rate:.2f}%)")

In [ ]:
# Visualisation de la répartition des pannes
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Répartition Normal vs Panne', 'Types de Pannes', 'Distribution par Type de Produit', 'Évolution Temporelle'],
    specs=[[{"type": "pie"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "scatter"}]]
)

# 1. Pie chart Normal vs Panne
normal_count = len(df[df['Machine failure'] == 0])
failure_count = len(df[df['Machine failure'] == 1])

fig.add_trace(
    go.Pie(labels=['Normal', 'Panne'], values=[normal_count, failure_count],
           marker_colors=['lightblue', 'red']),
    row=1, col=1
)

# 2. Bar chart par type de panne
failure_counts = [df[ft].sum() for ft in failure_types]
fig.add_trace(
    go.Bar(x=failure_types, y=failure_counts, marker_color='orange'),
    row=1, col=2
)

# 3. Distribution par type de produit
product_failure = df.groupby('Type')['Machine failure'].agg(['count', 'sum']).reset_index()
product_failure['failure_rate'] = product_failure['sum'] / product_failure['count'] * 100

fig.add_trace(
    go.Bar(x=product_failure['Type'], y=product_failure['failure_rate'], 
           marker_color='green', name='Taux de panne %'),
    row=2, col=1
)

# 4. Évolution temporelle (simulation avec UDI)
df_sample = df.iloc[::100]  # Échantillonnage pour la visualisation
fig.add_trace(
    go.Scatter(x=df_sample['UDI'], y=df_sample['Machine failure'], 
               mode='markers', marker_color='red', name='Pannes'),
    row=2, col=2
)

fig.update_layout(height=800, title_text="📊 Vue d'ensemble des Pannes Industrielles")
fig.show()

In [ ]:
# Analyse des variables capteurs
sensor_vars = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

print("🌡️ ANALYSE DES VARIABLES CAPTEURS")
print("=" * 60)

# Statistiques descriptives
sensor_stats = df[sensor_vars].describe()
display(sensor_stats)

# Comparaison Normal vs Panne
print("\n⚡ COMPARAISON NORMAL vs PANNE")
print("=" * 60)

comparison = pd.DataFrame({
    'Normal_Mean': df[df['Machine failure'] == 0][sensor_vars].mean(),
    'Failure_Mean': df[df['Machine failure'] == 1][sensor_vars].mean(),
    'Normal_Std': df[df['Machine failure'] == 0][sensor_vars].std(),
    'Failure_Std': df[df['Machine failure'] == 1][sensor_vars].std()
})

comparison['Difference'] = comparison['Failure_Mean'] - comparison['Normal_Mean']
comparison['Diff_Pct'] = (comparison['Difference'] / comparison['Normal_Mean']) * 100

display(comparison)

In [ ]:
# Distributions des variables capteurs - Normal vs Panne
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, var in enumerate(sensor_vars):
    if i < len(axes):
        # Distribution normale
        axes[i].hist(df[df['Machine failure'] == 0][var], alpha=0.7, label='Normal', 
                    color='lightblue', bins=50, density=True)
        
        # Distribution avec panne
        axes[i].hist(df[df['Machine failure'] == 1][var], alpha=0.7, label='Panne', 
                    color='red', bins=50, density=True)
        
        axes[i].set_title(f'Distribution: {var}')
        axes[i].set_xlabel(var)
        axes[i].set_ylabel('Densité')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

# Masquer le dernier subplot vide
axes[-1].set_visible(False)

plt.tight_layout()
plt.suptitle('🌡️ Distributions des Variables Capteurs : Normal vs Panne', fontsize=16, y=1.02)
plt.show()

In [ ]:
# Analyse de corrélation
print("🔗 MATRICE DE CORRÉLATION")
print("=" * 50)

# Calculer la matrice de corrélation
correlation_vars = sensor_vars + ['Machine failure'] + failure_types
corr_matrix = df[correlation_vars].corr()

# Visualisation avec heatmap
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdYlBu_r', center=0,
            square=True, fmt='.2f', cbar_kws={"shrink": .8})
plt.title('🔗 Matrice de Corrélation - Capteurs et Pannes')
plt.tight_layout()
plt.show()

# Top corrélations avec les pannes
print("\n🎯 CORRÉLATIONS FORTES AVEC LES PANNES")
print("=" * 50)

failure_corr = corr_matrix['Machine failure'].abs().sort_values(ascending=False)
print("Corrélations avec 'Machine failure':")
for var, corr in failure_corr.items():
    if var != 'Machine failure' and corr > 0.1:
        print(f"  {var:<30}: {corr:.3f}")

In [ ]:
# Analyse spécifique par type de panne
print("🔥 ANALYSE PAR TYPE DE PANNE")
print("=" * 60)

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for i, failure_type in enumerate(failure_types):
    if i < len(axes):
        # Données pour ce type de panne
        failure_data = df[df[failure_type] == 1]
        normal_data = df[df['Machine failure'] == 0]
        
        # Scatter plot des variables les plus corrélées
        if failure_type == 'TWF':  # Tool Wear Failure
            x_var, y_var = 'Tool wear [min]', 'Torque [Nm]'
        elif failure_type == 'HDF':  # Heat Dissipation Failure
            x_var, y_var = 'Air temperature [K]', 'Process temperature [K]'
        elif failure_type == 'PWF':  # Power Failure
            x_var, y_var = 'Torque [Nm]', 'Rotational speed [rpm]'
        elif failure_type == 'OSF':  # Overstrain Failure
            x_var, y_var = 'Torque [Nm]', 'Tool wear [min]'
        else:  # RNF - Random Failure
            x_var, y_var = 'Process temperature [K]', 'Rotational speed [rpm]'
        
        # Plot normal data
        axes[i].scatter(normal_data[x_var], normal_data[y_var], 
                       alpha=0.3, c='lightblue', s=1, label='Normal')
        
        # Plot failure data
        axes[i].scatter(failure_data[x_var], failure_data[y_var], 
                       alpha=0.8, c='red', s=10, label=f'{failure_type} Failure')
        
        axes[i].set_xlabel(x_var)
        axes[i].set_ylabel(y_var)
        axes[i].set_title(f'{failure_names[failure_type]}\n({len(failure_data)} cas)')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

# Masquer le dernier subplot vide
axes[-1].set_visible(False)

plt.tight_layout()
plt.suptitle('🔥 Patterns de Pannes par Type', fontsize=16, y=1.02)
plt.show()

In [ ]:
# Extraction des seuils d'alerte
print("⚠️ SEUILS D'ALERTE RECOMMANDÉS")
print("=" * 60)

# Calculer les percentiles pour chaque variable
normal_data = df[df['Machine failure'] == 0]
failure_data = df[df['Machine failure'] == 1]

thresholds = {}

for var in sensor_vars:
    normal_95 = normal_data[var].quantile(0.95)
    normal_5 = normal_data[var].quantile(0.05)
    failure_median = failure_data[var].median()
    
    # Déterminer le type de seuil selon la variable
    if var in ['Air temperature [K]', 'Process temperature [K]', 'Tool wear [min]']:
        # Seuil supérieur (alerte si valeur > seuil)
        threshold = normal_95
        alert_type = "Seuil MAX"
    elif var == 'Rotational speed [rpm]':
        # Seuils min et max
        threshold_min = normal_5
        threshold_max = normal_95
        threshold = f"{threshold_min:.1f} - {threshold_max:.1f}"
        alert_type = "Plage normale"
    else:  # Torque
        threshold = normal_95
        alert_type = "Seuil MAX"
    
    thresholds[var] = {
        'threshold': threshold,
        'type': alert_type,
        'normal_median': normal_data[var].median(),
        'failure_median': failure_median
    }
    
    print(f"{var}:")
    print(f"  {alert_type:<15}: {threshold}")
    print(f"  Normal (médiane): {normal_data[var].median():.2f}")
    print(f"  Panne (médiane) : {failure_median:.2f}")
    print()

# Sauvegarder les seuils pour le simulateur
import json
thresholds_json = {var: {'threshold': float(data['threshold']) if isinstance(data['threshold'], (int, float)) else data['threshold'],
                        'type': data['type']} for var, data in thresholds.items()}

with open('../data/processed/alert_thresholds.json', 'w') as f:
    json.dump(thresholds_json, f, indent=2)

print("💾 Seuils sauvegardés dans data/processed/alert_thresholds.json")

In [ ]:
# Analyse des scénarios de panne extraits
print("🎬 ANALYSE DES SCÉNARIOS DE PANNE")
print("=" * 60)

# Statistiques des scénarios
print(f"Total scénarios extraits: {len(scenarios)}")
print(f"Répartition par type:")
print(scenarios['failure_type'].value_counts())

print(f"\nRépartition par sévérité:")
print(scenarios['severity'].value_counts())

print(f"\nExemples de scénarios par type:")
print("=" * 40)

for failure_type in scenarios['failure_type'].unique():
    scenario_examples = scenarios[scenarios['failure_type'] == failure_type].head(2)
    print(f"\n{failure_type} - {failure_names.get(failure_type, failure_type)}:")
    for _, row in scenario_examples.iterrows():
        print(f"  Scénario {row['scenario_id']}: {row['description']} (Sévérité: {row['severity']})")

In [ ]:
# Insights finaux et recommandations pour le hackathon
print("🎯 INSIGHTS CLÉS POUR LE HACKATHON IoT")
print("=" * 70)

insights = [
    f"📊 Dataset: {len(df):,} échantillons avec {df['Machine failure'].mean()*100:.1f}% de pannes",
    f"🔥 {len(scenarios)} scénarios réalistes extraits pour simulation",
    f"🌡️ Variables critiques identifiées:",
    f"   • Tool wear: Facteur #1 pour TWF (usure d'outil)", 
    f"   • Température: Critique pour HDF (dissipation thermique)",
    f"   • Couple/Vitesse: Indicateurs PWF/OSF (puissance/contrainte)",
    f"⚠️ Seuils d'alerte calculés pour chaque capteur",
    f"🤖 Features prêtes pour entraînement ML",
    f"🎬 Scénarios variés: {scenarios['severity'].value_counts().to_dict()}"
]

for insight in insights:
    print(insight)

print("\n🚀 PRÊT POUR LA SUITE :")
print("=" * 30)
print("✅ 1. Données analysées et comprises")
print("✅ 2. Seuils d'alerte définis")
print("✅ 3. Scénarios de simulation extraits")
print("🔜 4. Développer le simulateur IoT")
print("🔜 5. Créer le modèle de détection d'anomalies")
print("🔜 6. Implémenter le dashboard temps réel")

print("\n💡 Cette analyse vous donne une base solide de données réelles pour votre hackathon !")